# Evaluation pipeline demo


This is a short notebook showing how the evaluation pipeline works WITHOUT having to download the entire dataset and configure the container. Small subsets of each dataset is included in duckdb files in this folder.

Ensure you have Python 3 and `uv`, then just run `uv sync` from the repository root and ensure the notebook is connected to that virtual environment (likely `.venv` by default)

## Datasets
1. [Risk-Based Authentication (RBA)](https://zenodo.org/records/6782156) (Wiefling et al, 2022)
    - synthetic dataset of 31m authentication records from ~3m users with user IDs from an SSO service
    - includes user agent, IP, etc., as well as fields for (a) whether the login originates from a known attack IP address or (b) the login was flagged as an account takeover (ground truth)
    - Licensed by the authors (Wiefling et al.) under a [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

2. [FP-Stalker](https://github.com/Spirals-Team/FPStalker) (Vastel et al., 2018) 
    - 15k record sample from the AmIUnique dataset from 2015-2017. 
    - Each record contains a long term tracking cookie (tying it to other records from the same origin) alongside a full user agent string and other fingerprint attributes we do not use in our ER method (e.g., canvas).
    - Licensed by the authors (Vastel et al.) under the [GNU-AGPL 3.0 License](https://github.com/Spirals-Team/FPStalker/blob/master/LICENSE)



The datasets provided in this demo are small subsets of the above datasets just to show how this works at a high level:
1. RBA subset (3.2k rows): Selected only only "User IDs" that have at least one row flagged as "Is Account Takeover"
2. FP Stalker Subset (8.6k rows): Selected only tracking cookie IDs that were live in June 2016. Also removed some columns that aren't useful to us.


In [ ]:
import duckdb
import pandas as pd

with duckdb.connect('./trunc_rba.duckdb') as rba_conn:
    rba_df_orig = rba_conn.execute("SELECT * FROM imported_data").df()

print("Length of RBA truncated dataset:", rba_df_orig.shape[0])
display(rba_df_orig.head())

In [ ]:
with duckdb.connect('./trunc_fp_stalker.duckdb') as fp_conn:
    fp_df_orig = fp_conn.execute("SELECT * FROM imported_data").df() 

print("Length of FP Stalker truncated dataset:", fp_df_orig.shape[0])
display(fp_df_orig.head())

# Parse User Agent Strings 

Parse the user agent strings (e.g., `Mozilla/5.0 (Windows NT 6.1; WOW64; rv:41.0) Gecko/20100101 Firefox/41.0`) using the same `field_normalization` modules as the webapp, then move them into the column names expected by the `device_grouping2` module.

In [ ]:
import sys
from pathlib import Path
import tqdm

_root = Path.cwd().parent.parent
_python_core = str(_root / "python_core")
if _python_core not in sys.path:
    sys.path.append(_python_core)
if str(_root) not in sys.path:
    sys.path.append(str(_root))

from python_core.field_normalization.user_agent import UserAgentParser
from python_core.field_normalization.device import normalize_device_fields

In [ ]:
ua_parser = UserAgentParser()

def normalize_data(df: pd.DataFrame, 
                   user_agent_col_name: str, 
                   timestamp_col_name: str,
                   other_cols_to_keep: list = []) -> pd.DataFrame:
    dicts = []
    for _, row in tqdm.tqdm(df.iterrows(), total=len(df)):
        d = {"user_agent_original": row.get(user_agent_col_name, "")}
        d = ua_parser.parse(d) # returns dict
        d = normalize_device_fields(d)
        d = {k: v for k, v in d.items() if k.startswith("norm__")}
        d['timestamp'] = row.get(timestamp_col_name, "")
        dicts.append(d)
    
    new_columns = pd.DataFrame(dicts, index=df.index)
    normalized_cols = [c for c in new_columns.columns if c.startswith("norm__")]
    
    if isinstance(other_cols_to_keep, list):
        other_cols_to_keep = [c for c in other_cols_to_keep if c in df.columns]
    else:
        other_cols_to_keep = []
    
    new_df = df[other_cols_to_keep + [user_agent_col_name]]
    new_df = pd.concat([
        new_columns[["timestamp"] + normalized_cols], 
        new_df], axis=1)
    new_df.rename(columns={c: f"attr__{c}" for c in normalized_cols}, inplace=True)
    return new_df

In [ ]:
s = "Mozilla/5.0  (iPhone; CPU iPhone OS 11_2_6 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko Version/4.0 Chrome/85.0.4183.127 Mobile Safari/537.36Snapchat11.1.7.81 (LYA-L29; Android 10#10.1.0.288C432#29; gzip"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/69.0.3497.17.19.92 Safari/537.36"
s = "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.1 Safari/605.1.15"
s = "Mozilla/5.0  (iPad; CPU OS 7_1 like Mac OS X) AppleWebKit/533.1 (KHTML, like Gecko Version/4.0 Mobile Safari/533.1 variation/277457"
ua_parser.parse({"user_agent_original": s})

In [ ]:
rba_df_orig = rba_df_orig[rba_df_orig['Login Successful'] == True]
norm_rba_df = normalize_data(rba_df_orig,
                             user_agent_col_name='User Agent String', 
                             timestamp_col_name='Login Timestamp',
                             other_cols_to_keep=['index', 'User ID', 'Login Successful', 'Is Account Takeover', 'Is Attack IP', 'Browser Name and Version', 'OS Name and Version'])                           

norm_rba_df.rename(columns={'index': 'id'},  inplace=True)  # "id" has to be the unique id of the row
norm_rba_df.convert_dtypes()
norm_rba_df

In [ ]:
norm_fp_df = normalize_data(fp_df_orig,
                            user_agent_col_name='userAgentHttp', 
                            timestamp_col_name='creationDate',
                            other_cols_to_keep=['id', 'counter', 'osDetailed', 'browserDetailed'])

norm_fp_df.rename(columns={'id': 'tracking_id'},  inplace=True)  # "id" has to be the unique id of the row
norm_fp_df.rename(columns={'counter': 'id'}, inplace=True)

norm_fp_df.convert_dtypes()
norm_fp_df

# RBA Heuristic Evaluation

We evaluate our device grouping heuristic using the dataset from:
> Stephan Wiefling, Paul René Jørgensen, Sigurd Thunem, and Luigi Lo Iacono. 2022. **Pump Up Password Security! Evaluating and Enhancing Risk-Based Authentication on a Real-World Large-Scale Online Service**. *ACM Transactions on Privacy and Security (TOPS)*. [DOI: 10.1145/3549079](https://doi.org/10.1145/3549079) | [Zenodo Dataset](https://doi.org/10.5281/zenodo.6782155)

The dataset contains synthetic values for privacy. Each record includes:
- User account ID (`User ID`)
- User agent and metadata fields (`User Agent String`)
- Account takeover flag (`Is Account Takeover` - ground-truth compromise)
- Known attack IP flag (`Is Attack IP` - ground-truth indicator of compromise)

For all user accounts with at least one flagged account takeover, we group their logins into device instances and evaluate:
1. Do we accidentally group non_takeover logins with takeover logins?
2. Are these errors caused by spoofed user agents matching non_takeover ones?

In [ ]:
from python_core.device_grouping2 import client_os_upgrades, profiles
from python_core.device_grouping2.instances import DeviceInstanceGraph
import datetime
import json

In [ ]:
start_time  = datetime.datetime.now().isoformat()
norm_rba_df['predicted_instance_id'] = None

for user_id in map(int, norm_rba_df['User ID'].dropna().unique()):
    user_df = norm_rba_df[norm_rba_df['User ID'] == user_id]
    upgrade_edges = client_os_upgrades.get_edges(user_df)
    graph = DeviceInstanceGraph(user_df, pd.DataFrame(upgrade_edges))

    for i, inst in enumerate(graph.get_instances()):
        norm_rba_df.loc[inst.df.index, 'predicted_instance_id'] = f"{user_id}_inst_{i}"


In [ ]:
gp = norm_rba_df.groupby('predicted_instance_id')
sizes = gp.size()

summary = {
    "run": {
        "description": "Truncated RBA dataset for demo",
        "start_time": start_time,
    },
    "dataset": {
        "num_records": len(norm_rba_df),
        "num_users": len(norm_rba_df['User ID'].unique()),
        "num_instances": {
            "total": len(gp),
            "singletons": int((sizes == 1).sum()),
            "median_size": int(sizes.median()),
            "mean_size": float(sizes.mean()),
            "max_size": int(sizes.max()),
        }
    }
}

for key, col in [
    ("takeover", "Is Account Takeover"),
    ("attack_ip", "Is Attack IP")
]:
    is_flag = norm_rba_df[col] == 1
    is_unflag = norm_rba_df[col] == 0
    
    has_flag = is_flag.groupby(norm_rba_df['predicted_instance_id']).any()
    has_unflag = is_unflag.groupby(norm_rba_df['predicted_instance_id']).any()
    
    inst_type = pd.Series("only_unflagged", index=has_flag.index)
    inst_type[has_flag & has_unflag] = "mixed"
    inst_type[has_flag & ~has_unflag] = "only_flagged"
    norm_rba_df[f'{key}_instance_type'] = norm_rba_df['predicted_instance_id'].map(inst_type)
    
    mixed_ids = has_flag[has_flag & has_unflag].index
    mixed_df = norm_rba_df[norm_rba_df['predicted_instance_id'].isin(mixed_ids)]
    
    mixed_identical_ua = 0
    mixed_heuristic_error = 0
    
    if not mixed_df.empty:
        for _, inst_df in mixed_df.groupby('predicted_instance_id'):
            clean_uas = set(inst_df[inst_df[col] == 0]['User Agent String'].dropna())
            flagged_uas = set(inst_df[inst_df[col] == 1]['User Agent String'].dropna())
            
            # given instance (cluster) C
            if flagged_uas & clean_uas:
                mixed_identical_ua += 1  # At least one flagged login in C has the same UA as a clean login in C
            
            if flagged_uas - clean_uas: # At least one flagged login in C has a UA that does not match any clean login in C.
                mixed_heuristic_error += 1
                
    summary[key] = {
        "num_records": {
            "total": len(norm_rba_df),
            "flagged": int(is_flag.sum()),
            "unflagged": int(is_unflag.sum())
        },
        "num_instances": {
            "total": len(gp),
            "containing_only_flagged": int((has_flag & ~has_unflag).sum()),
            "containing_only_unflagged": int((~has_flag & has_unflag).sum()),
            "containing_mixed": {
                "total": len(mixed_ids),
                "containing_identical_ua": mixed_identical_ua,
                "containing_heuristic_error": mixed_heuristic_error
            }
        }
    }


In [ ]:
_start_time = start_time.replace(":", "-").replace(".", "-")
dirpath =  Path("runs") / f"rba_{_start_time}"
dirpath.mkdir(parents=True, exist_ok=True)

summary["run"]["end_time"] = datetime.datetime.now().isoformat()
summary["run"]["duration"] = str(datetime.datetime.fromisoformat(summary["run"]["end_time"]) - datetime.datetime.fromisoformat(start_time))

with open(dirpath / "summary.json", "w") as f:
    json.dump(summary, f, indent=4)

cols_to_save = [
    'User ID',
    'predicted_instance_id',
    'id',
    'timestamp',
    'Is Account Takeover',
    'takeover_instance_type',
    'Is Attack IP',
    'attack_ip_instance_type',
    'User Agent String',
    'attr__norm__client_name',
    'attr__norm__client_version',
    'attr__norm__os_name',
    'attr__norm__os_version',
    'attr__norm__model_name',
    'attr__norm__manufacturer'
]
norm_rba_df[cols_to_save].to_csv(dirpath / "logins_with_predictions.csv", index=False)
